In [1]:
# Load necessary Libraries
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
import faiss
import torch

In [2]:
# Load Corpus (docs)
df = pd.read_csv("rag_docs/corpus_2025.csv")
texts = df["text"].fillna("").tolist()

In [3]:
# List of Questions
questions = [
    "What was Marquette's hitting percentage in 2025?",
    "Which Big East team had the most digs?",
    "List the setters on Creighton’s roster.",
    "Who were the best five hitters in the Big Ten, based on kills and hitting percentage?",
    "Who is likely to win if Kentucky and Florida play each other?",
    "Which player in the Big 12 had the most kills?",
    "Which team(s) ended the season with the best winning record?",
    "What was Stanford's blocking average?",
    "How many service aces did Texas A&M have?",
    "How did Wisconsin perform in five-set matches?",
    "Which Summit League tea had the highest hitting efficiency?",
    "Rank the ACC teams by total blocks.",
    "List the liberos on Louisville's roster",
    "Who was the starting middle blocker for Marquette?",
    "Who were the top three servers in the ACC based on aces?",
    "Compare the hitting percentages of the top hitters from Baylor and BYU.",
    "How did the top Big 12 setter compare to the top Big Ten setter?",
    "If Stanford played Nebraska, which team would statistically have the advantage?",
    "Based on 2025 stats, who would likely win between Pitt and Louisville?",
    "Which team would be favored in a matchup between Texas and Wisconsin?",
    "Which players most likely were named to the 2025 All‑American First Team?"
]

In [4]:
# Chunking
def chunk_text(text, chunk_size=800, overlap=200):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk.strip())
        start = end - overlap
        if start < 0:
            start = 0
    return [c for c in chunks if c]

kb_chunks = []
for t in texts:
    kb_chunks.extend(chunk_text(t))

print("Total chunks:", len(kb_chunks))

Total chunks: 390


In [5]:
# Embeddings + faiss
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

kb_embeddings = embed_model.encode(
    kb_chunks,
    convert_to_numpy=True,
    show_progress_bar=True
).astype("float32")

dim = kb_embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(kb_embeddings)

print("FAISS index size:", index.ntotal)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/13 [00:00<?, ?it/s]

FAISS index size: 390


In [6]:
import re


def clean_text(text):
    text = re.sub(r"<chr>|<dbl>|chr>|dbl>", "", text)
    text = re.sub(r"#.*tibble.*", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def format_player(row):
    return (
        f"{row['player']} is a {row['pos']} for {row['team']} "
        f"in {row['conference']} ({row['season']}). "
        f"Stats: {row['kills']} kills, {row['assists']} assists, {row['aces']} aces."
    )

In [7]:
texts = [clean_text(t) for t in texts]
player_texts = player_df.apply(format_player, axis=1).tolist()

In [8]:
def is_valid_chunk(text):
    bad_patterns = ["chr>", "dbl>", "tibble", "<chr", "<dbl"]
    return not any(p in text for p in bad_patterns)

In [9]:
kb_chunks = [c for c in kb_chunks if is_valid_chunk(c)]
kb_chunks = df["text"].dropna().astype(str).tolist()
print("Clean KB samples:")
for c in kb_chunks[:5]:
    print("----")
    print(c)
kb_embeddings = embed_model.encode(kb_chunks).astype("float32")
index.reset()
index.add(kb_embeddings)

Clean KB samples:
----
Team: Chattanooga
Conference: SoCon
Season: 2025
TeamID: 604920

Team Season Stats:
# A tibble: 1 × 21
  season    teamid team        conference     s kills errors
  <chr>     <chr>  <chr>       <chr>      <dbl> <dbl>  <dbl>
1 2025-2026 604920 Chattanooga SoCon        111  1391    562
# ℹ 14 more variables: total_attacks <dbl>, hit_pct <dbl>,
#   assists <dbl>, aces <dbl>, serr <dbl>, digs <dbl>,
#   retatt <dbl>, rerr <dbl>, block_solos <dbl>,
#   block_assists <dbl>, berr <dbl>, pts <dbl>, bhe <dbl>,
#   trpl_dbl <dbl>

Player Stats:
# A tibble: 16 × 28
   season   teamid team  conference number player yr    pos  
   <chr>    <chr>  <chr> <chr>       <dbl> <chr>  <chr> <chr>
 1 2025-20… 604920 Chat… SoCon           0 Baile… Jr    MB   
 2 2025-20… 604920 Chat… SoCon           1 Addis… Fr    OH   
 3 2025-20… 604920 Chat… SoCon           4 Mallo… So    S    
 4 2025-20… 604920 Chat… SoCon           5 Jordi… Fr    MB   
 5 2025-20… 604920 Chat… SoCon           6 

In [10]:
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def generate_no_rag(question):
    prompt = f"Answer the question: {question}"
    
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    
    outputs = model.generate(
        **inputs,
        max_length=100
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def retrieve(query, top_k=3):
    query_embedding = embed_model.encode([query]).astype("float32")
    
    faiss.normalize_L2(query_embedding)
    
    distances, indices = index.search(query_embedding, top_k)
    
    results = [kb_chunks[i] for i in indices[0]]
    
    return results

def generate_rag(question):
    retrieved_docs = retrieve(question, top_k=10)
    context = " ".join(retrieved_docs)

    ####################first prompt used
    # prompt = f"""
    # Use the following context to answer the question.

    # Context:
    # {context}

    # Question: {question}
    # Answer:
    # """

    ###############second prompt used
    # prompt = f"""
    # You are analyzing volleyball statistics.

    # Use the context to answer the question as best as possible.
    # If exact numbers are unclear, estimate based on the data.

    # Context:
    # {context}

    # Question: {question}

    # Answer:
    # """
    ############## third prompt used
    # prompt=f"""
    # You are an expert in analyzing volley statistics answering questions in simple ways for beginning players or fans.

    # Use the context to answer the question as best as possible, with simple language anyone can understand.
    # If exact numbers are unclear, estimate based on the data.

    # Reframe the question into the answer so that its easier to understand

    # Context:
    # {context}

    # Question: {question}

    # Answer:
    # """
    ################4th prompt used
    prompt=f"""
    You are an expert in analyzing volley statistics answering questions in simple ways for beginning players or fans.

    Use the context to answer the question as best as possible, with simple language anyone can understand.
    If exact numbers are unclear, estimate based on the data.

    Context:
    {context}

    Question: {question}

    Answer:
    """
    
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    
    outputs = model.generate(
        **inputs,
        max_length=150
    )
    
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    return answer, retrieved_docs

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [11]:
results = []

for q in questions:
    no_rag = generate_no_rag(q)
    rag_answer, docs = generate_rag(q)
    
    results.append({
        "Question": q,
        "No RAG": no_rag,
        "RAG": rag_answer,
        "Evidence": docs
    })

df_results = pd.DataFrame(results)
df_results.to_csv("rag_results4.csv", index=False)

Question: What was Marquette's hitting percentage in 2025?,
    No RAG: .22%,
    RAG: Gonzaga WCC 112 1430 591,
    
Question: Which Big East team had the most digs?,
    No RAG: the New England Patriots,
    RAG: 106 1394 478,
    
Question: List the setters on Creighton’s roster.,
    No RAG: robert mccartney jr. , jim mccartney , jim mccartney , jim mccartney , jim mccartney , jim mccartney , jim mccartney , jim mccartney , jim mc,
    RAG:                                                                           ,
    
Question: Who were the best five hitters in the Big Ten, based on kills and hitting percentage?,
    No RAG: adam sandler,
    RAG:                                                                           ,
    
Question: Who is likely to win if Kentucky and Florida play each other?,
    No RAG: Florida,
    RAG:                                                                           ,
    
Question: Which player in the Big 12 had the most kills?,
    No RAG: Bil

In [12]:
for r in results:
    print("\n==============================")
    print("Q:", r["Question"])
    print("No RAG:", r["No RAG"])
    print("RAG:", r["RAG"])
    
    print("\nRetrieved Evidence:")
    for doc in r["Evidence"]:
        print("-", doc[:200])


Q: What was Marquette's hitting percentage in 2025?
No RAG: .22%
RAG: Gonzaga WCC 112 1430 591

Retrieved Evidence:
- Team: Gonzaga
Conference: WCC
Season: 2025
TeamID: 604828

Team Season Stats:
# A tibble: 1 × 21
  season    teamid team    conference     s kills errors
  <chr>     <chr>  <chr>   <chr>      <dbl> <d
- Team: James Madison
Conference: Sun Belt
Season: 2025
TeamID: 604851

Team Season Stats:
# A tibble: 1 × 21
  season    teamid team         conference     s kills errors
  <chr>     <chr>  <chr>      
- Team: New Orleans
Conference: Southland
Season: 2025
TeamID: 604900

Team Season Stats:
# A tibble: 1 × 21
  season    teamid team        conference     s kills errors
  <chr>     <chr>  <chr>       <
- Team: Arkansas St.
Conference: Sun Belt
Season: 2025
TeamID: 604752

Team Season Stats:
# A tibble: 1 × 21
  season    teamid team         conference     s kills errors
  <chr>     <chr>  <chr>       
- Team: Louisiana
Conference: Sun Belt
Season: 2025
TeamID: 604911

Tea

In [57]:
print(df.columns)

Index(['doc_id', 'conference', 'season', 'team', 'team_id', 'doc_type',
       'text'],
      dtype='object')
